# Principal Component Analysis (PCA)

**Topic:** Unsupervised Learning — Dimensionality Reduction

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import Dropdown, IntSlider, FloatSlider, Output, HBox, VBox
from IPython.display import display, clear_output, Markdown
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from tkh_utils import PALETTE, FONT, base_layout, load_california_housing


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** what principal components are and how they relate to variance in the data
- **Explain** how PCA compresses many features into fewer dimensions while preserving most information
- **Interpret** a scree plot to choose how many components to keep

> **Tip:** In the scree plot widget, look for the "elbow" — the point where adding more components gives diminishing returns on variance explained. That elbow is typically a good choice for the number of components to keep.

---
## How we got here

In **[feature_engineering/08_dimensionality_reduction_intro.ipynb](../feature_engineering/08_dimensionality_reduction_intro.ipynb)** you saw a brief introduction to PCA. In **[math/linear_algebra/07_eigenvalues_and_eigenvectors.ipynb](../math/linear_algebra/07_eigenvalues_and_eigenvectors.ipynb)** you learned how eigendecomposition finds the principal directions in a matrix. In **[math/linear_algebra/08_vector_spaces_and_projections.ipynb](../math/linear_algebra/08_vector_spaces_and_projections.ipynb)** you learned how to project vectors onto new bases.

PCA is the applied intersection of those two concepts: eigendecomposition of the covariance matrix gives you the directions of maximum variance, and projection onto those directions gives you the compressed representation.

---
## Why this matters for data science

Datasets with 100, 1000, or 100,000 features are not unusual. Visualizing them is impossible. Training on them is slow. Many features are noisy or correlated, adding variance without adding signal.

PCA gives you a compact representation that preserves most of the information using only a few dimensions. It is a standard preprocessing step for visualization, clustering, noise reduction, and as input to downstream models.

In **[ml_concepts/11_the_curse_of_dimensionality.ipynb](../ml_concepts/11_the_curse_of_dimensionality.ipynb)** you learned why distances and neighborhoods stop being meaningful as dimensions grow. Dimensionality reduction is the practical response: compress down to the dimensions that actually carry signal before distance-based methods, models, or your own eyes have to deal with the rest.

---
## Where it sits on the spectrum

Referencing **[ml_concepts/13_interpretability_vs_complexity.ipynb](../ml_concepts/13_interpretability_vs_complexity.ipynb)**:

**Interpretability: Medium.** Each principal component is a linear combination of the original features. The first few components can sometimes be interpreted (e.g., "component 1 is mostly income-related features"), but they are not as directly readable as the original features.

**Complexity: Low.** Time complexity O(min(n, d)²) where n = samples and d = features. Very fast in practice thanks to sklearn's optimized SVD implementations.

**Position:** Medium interpretability, low complexity — the workhorse of dimensionality reduction. Use it as a first step before trying non-linear methods.

---
## How it learns

PCA finds the directions of maximum variance in your data.

**Step 1 — Center and scale.** Subtract the mean from each feature (centering). Optionally divide by standard deviation (scaling, recommended for features on different scales).

**Step 2 — Compute covariance.** Build the covariance matrix that captures how each pair of features varies together.

**Step 3 — Eigendecomposition.** Find the eigenvectors and eigenvalues of the covariance matrix. Each eigenvector is a principal component direction; each eigenvalue is the variance along that direction.

**Step 4 — Sort and select.** Sort eigenvectors by eigenvalue (highest first). Keep the top k eigenvectors. These k directions capture the most variance.

**Step 5 — Project.** Multiply your data by the k selected eigenvectors to get the k-dimensional compressed representation.

---
## The math behind it

**Covariance matrix** (after centering):
$$\mathbf{C} = \frac{1}{n-1} X^T X$$
Where $X$ is the mean-centered data matrix ($n$ samples × $d$ features).

**Eigendecomposition** finds vectors $\mathbf{v}$ and scalars $\lambda$ satisfying:
$$\mathbf{C} \mathbf{v} = \lambda \mathbf{v}$$
Each eigenvector $\mathbf{v}_k$ is a principal component direction. Each eigenvalue $\lambda_k$ is the variance along that direction.

**Variance explained** by component $k$:
$$\text{VE}_k = \frac{\lambda_k}{\sum_{j=1}^{d} \lambda_j}$$

**Projection** onto the top $k$ components:
$$\mathbf{Z} = X \mathbf{V}_k$$
Where $\mathbf{V}_k$ is the $d \times k$ matrix of the top $k$ eigenvectors. $\mathbf{Z}$ has shape $n \times k$: the compressed representation.

In practice, sklearn uses SVD (Singular Value Decomposition) instead of explicit eigendecomposition — numerically more stable and faster for large matrices.

---
## Try it yourself

In [ ]:
out1 = Output()

n_comp_slider = IntSlider(
    value=2, min=1, max=8, step=1,
    description="Components kept:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="420px"),
)

X, y = load_california_housing()
X_sc = StandardScaler().fit_transform(X)
price_tier = pd.qcut(y, q=3, labels=["Low", "Mid", "High"])
feature_names = list(X.columns)

pca_all = PCA(random_state=42).fit(X_sc)
var_exp = pca_all.explained_variance_ratio_
cum_var = np.cumsum(var_exp)
n90 = int(np.argmax(cum_var >= 0.90) + 1)

def render_scree(n_keep):
    bar_colors = [PALETTE["primary"] if i < n_keep else PALETTE["muted"] for i in range(len(var_exp))]
    traces = [
        go.Bar(
            x=[f"PC{i+1}" for i in range(len(var_exp))], y=var_exp,
            marker_color=bar_colors,
            text=[f"{v:.1%}" for v in var_exp], textposition="outside",
            name="Variance explained",
        ),
        go.Scatter(
            x=[f"PC{i+1}" for i in range(len(var_exp))], y=cum_var,
            mode="lines+markers",
            line=dict(color=PALETTE["secondary"], width=2, dash="dot"),
            name="Cumulative variance", yaxis="y2",
        ),
    ]
    layout = base_layout(
        title=f"Scree Plot — {n_keep} Component{'s' if n_keep != 1 else ''} Capture{'s' if n_keep == 1 else ''} {cum_var[n_keep-1]:.1%} of Variance",
        xaxis_title="Principal component",
        yaxis_title="Fraction of variance explained",
    )
    layout.update(
        yaxis2=dict(overlaying="y", side="right", title="Cumulative variance", range=[0, 1.05]),
        showlegend=True,
    )
    fig = go.Figure(data=traces, layout=layout)
    fig.add_vline(x=n_keep - 0.5, line=dict(color=PALETTE["muted"], width=1, dash="dash"))

    caption = (
        f"**Keeping {n_keep} component{'s' if n_keep != 1 else ''} captures "
        f"{cum_var[n_keep-1]:.1%} of the variance.** Reaching 90% takes {n90} of the 8 "
        f"components here — variance is spread fairly evenly across the first six rather "
        f"than dropping off sharply after one or two, so this dataset doesn't compress as "
        f"cleanly as a textbook example might suggest."
    )

    with out1:
        clear_output(wait=True)
        fig.show()
        display(Markdown(caption))

def on_change_scree(change):
    render_scree(n_comp_slider.value)

n_comp_slider.observe(on_change_scree, names="value")
display(VBox([n_comp_slider, out1]))
render_scree(n_comp_slider.value)

In [ ]:
out2 = Output()

pc_dropdown = Dropdown(
    options=[("PC1", 0), ("PC2", 1)],
    value=0,
    description="Inspect loadings for:",
    style={"description_width": "140px"},
    layout=widgets.Layout(width="420px"),
)

pca2 = PCA(n_components=2, random_state=42).fit(X_sc)
Z = pca2.transform(X_sc)
loadings = pca2.components_
var2 = pca2.explained_variance_ratio_
tier_colors = {"Low": PALETTE["primary"], "Mid": PALETTE["accent"], "High": PALETTE["secondary"]}

def render_biplot(pc_idx):
    fig = make_subplots(
        rows=1, cols=2, column_widths=[0.6, 0.4],
        subplot_titles=("PC1 vs. PC2, colored by price tier", f"Loading scores for PC{pc_idx + 1}"),
    )

    for tier, color in tier_colors.items():
        mask = (price_tier == tier).to_numpy()
        fig.add_trace(go.Scatter(
            x=Z[mask, 0], y=Z[mask, 1], mode="markers",
            marker=dict(color=color, size=5, opacity=0.5),
            name=f"{tier} price",
        ), row=1, col=1)

    scale = np.abs(Z).max() * 1.1
    for j, feat in enumerate(feature_names):
        fig.add_trace(go.Scatter(
            x=[0, loadings[0, j] * scale], y=[0, loadings[1, j] * scale],
            mode="lines+text", line=dict(color=PALETTE["muted"], width=1.5),
            text=["", feat], textposition="top center", showlegend=False,
        ), row=1, col=1)

    order = np.argsort(np.abs(loadings[pc_idx]))
    fig.add_trace(go.Bar(
        x=loadings[pc_idx][order],
        y=[feature_names[i] for i in order],
        orientation="h",
        marker_color=[PALETTE["secondary"] if v < 0 else PALETTE["primary"] for v in loadings[pc_idx][order]],
        showlegend=False,
    ), row=1, col=2)

    fig.update_xaxes(title_text=f"PC1 ({var2[0]:.1%} variance)", row=1, col=1)
    fig.update_yaxes(title_text=f"PC2 ({var2[1]:.1%} variance)", row=1, col=1)
    fig.update_xaxes(title_text="Loading score", row=1, col=2)
    fig.update_layout(
        base_layout(title="Biplot and Loading Scores", xaxis_title="", yaxis_title=""),
        height=420, showlegend=True,
    )

    top_idx = order[-1]
    top_feat = feature_names[top_idx]
    top_val = loadings[pc_idx][top_idx]

    caption = (
        f"**PC{pc_idx + 1} is dominated by {top_feat} (loading {top_val:+.2f}).** "
        f"Price tiers show no visible separation in this view regardless of which "
        f"component you inspect — the axes are shaped by geography and dwelling size, "
        f"not by income, which is the feature most directly tied to price."
    )

    with out2:
        clear_output(wait=True)
        fig.show()
        display(Markdown(caption))

def on_change_biplot(change):
    render_biplot(pc_dropdown.value)

pc_dropdown.observe(on_change_biplot, names="value")
display(VBox([pc_dropdown, out2]))
render_biplot(pc_dropdown.value)

---
## What's happening?

The scree plot widget reveals PCA's core trade-off: how many dimensions do you need to preserve enough information? Slide the components-kept control and watch the vertical marker move — the elbow, where bars drop off and the cumulative line flattens, marks where additional components stop paying for themselves.

The biplot widget shows what a 2-component compression of California housing's 8 features looks like. The three price tiers show no visible separation — their centers barely move, and on PC1 the "High" tier actually lands almost on top of "Low." That's because PC1 and PC2 are dominated by location (latitude/longitude) and dwelling size (rooms/bedrooms per household), not by income — the one feature most directly tied to price (it correlates with price at 0.69, but with PC1/PC2 at only 0.08–0.23). This is a useful lesson in itself: PCA finds the directions of maximum variance, not the directions most predictive of any particular outcome. A feature can get compressed away even though it matters most for what you actually care about. Switch the loadings dropdown between PC1 and PC2: each ranked bar chart shows which original features push that component in which direction — the "loading scores" view that makes PCA's compressed axes partially interpretable.

---
## Key hyperparameters

**`n_components`**: Number of dimensions to keep. Can be an integer or a float (float means "keep enough components to explain this fraction of variance"). `n_components=0.95` keeps the minimum number of components explaining 95% of variance.

**`svd_solver`** (default `'auto'`): Algorithm for decomposition. `'randomized'` is faster for large datasets; `'full'` is exact.

**`whiten`** (default `False`): If True, each component is divided by its standard deviation — useful when components will be used as input to a classifier that expects spherical features.

sklearn docs: [sklearn.decomposition.PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)

---
## Strengths and weaknesses

| Strength | Weakness |
|----------|----------|
| Fast and deterministic | Captures only linear structure |
| Removes correlated features | Components are harder to interpret than original features |
| Reduces noise (later components are often noise) | Requires scaling first — sensitive to feature scale |
| Excellent preprocessing for visualization | Assumes linear relationships between features |
| Reduces memory and training time | Cannot handle missing values |

---
## When to use it / When NOT to use it

| Use PCA when... | Do NOT use PCA when... |
|-----------------|------------------------|
| Features are correlated and you want to remove redundancy | Structure in data is non-linear |
| You need to visualize high-dimensional data | You need to interpret each component directly |
| You want to speed up downstream training | Missing values are present (impute first) |
| Noise reduction is a goal | Dataset is already low-dimensional |
| Features need to be decorrelated for a linear model | You need a faithful visualization of cluster structure (use t-SNE) |

---
## Key takeaway

> **PCA compresses many features into a few that explain most of the variance — scale first, look for the elbow in the scree plot, and remember that each component is a mix of the originals, not a single feature.**

---
*Next up: 08_tsne — where you go non-linear to reveal cluster structure that PCA's straight lines miss*